In [1]:
!pip install -q evaluate sacrebleu sacremoses accelerate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 1.0 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 867.8/867.8 kB 6.3 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 129.6/129.6 kB 5.6 MB/s eta 0:00:00


In [2]:
import os
import time
import shutil
import torch
import numpy as np
import pandas as pd
from tqdm import tqdm
from datasets import load_from_disk
from torch.utils.data import DataLoader
import evaluate
from transformers import (
    AutoModelForSeq2SeqLM,
    AutoTokenizer,
    DataCollatorForSeq2Seq
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
bleu_metric = evaluate.load("sacrebleu")

DEFAULT_VI2EN_CHECKPOINT = "/kaggle/input/models/khnhhngnguynhu/phomt-mariannmt/pytorch/vie2eng/2/vienv2.pt"

def load_checkpoint_weights(model, checkpoint_path, device):
    """Xử lý nạp trọng số từ file .pt hoặc thư mục zip giải nén trên Kaggle."""
    target_path = checkpoint_path
    
    # 1. Xử lý trường hợp Kaggle tự bung nén file .pt thành thư mục
    if os.path.isdir(checkpoint_path):
        repacked_path = "/kaggle/working/temp_repacked_ckpt.pt"
        if not os.path.exists(repacked_path):
            print(f"Phát hiện checkpoint dạng thư mục, đang đóng gói lại: {repacked_path}")
            archive = shutil.make_archive("/kaggle/working/temp_repacked_ckpt", format="zip", root_dir=checkpoint_path)
            os.rename(archive, repacked_path)
        target_path = repacked_path

    print(f"Đang nạp trọng số từ: {target_path}")
    checkpoint = torch.load(target_path, map_location=device)

    # 2. Bóc tách state_dict (dù lưu dạng full dict hay raw state_dict)
    if isinstance(checkpoint, dict) and "model_state_dict" in checkpoint:
        state_dict = checkpoint["model_state_dict"]
    elif isinstance(checkpoint, dict):
        state_dict = checkpoint
    else:
        raise ValueError("Định dạng checkpoint không hợp lệ.")

    # 3. Khử tiền tố 'module.' nếu checkpoint từng train qua DataParallel / DDP
    clean_state_dict = {
        (k[7:] if k.startswith("module.") else k): v
        for k, v in state_dict.items()
    }

    # 4. Gán trọng số vào model
    load_res = model.load_state_dict(clean_state_dict, strict=False)
    print(f"Nạp checkpoint thành công! (Missing keys: {len(load_res.missing_keys)}, Unexpected keys: {len(load_res.unexpected_keys)})")


def evaluate_model(
    model_name: str,
    dataset_path: str,
    checkpoint_path: str = None,   # Nếu None: tự động lấy DEFAULT_VI2EN_CHECKPOINT
    split_name: str = "test",
    batch_size: int = 64,
    max_length: int = 128,
    num_beams: int = 1,
    max_samples: int = None
):
    """
    Thực hiện sinh văn bản và tính SacreBLEU.
    - checkpoint_path=None: Tự động dùng DEFAULT_VI2EN_CHECKPOINT.
    - checkpoint_path='hf_base': Chạy thuần model gốc từ Hugging Face Hub.
    - checkpoint_path=<đường_dẫn_khác>: Nạp checkpoint chỉ định.
    """
    strategy_name = "Greedy Search" if num_beams == 1 else f"Beam Search (beams={num_beams})"

    # Xác định đường dẫn checkpoint thực tế
    actual_checkpoint = checkpoint_path
    if actual_checkpoint is None:
        actual_checkpoint = DEFAULT_VI2EN_CHECKPOINT

    print(f"\n" + "="*70)
    print(f"Model Base  : {model_name}")
    print(f"Checkpoint  : {actual_checkpoint}")
    print(f"Strategy    : {strategy_name}")
    print(f"Data Path   : {dataset_path} [{split_name}]")
    print(f"="*70 + "\n")

    # 1. Khởi tạo Tokenizer & Base Architecture
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

    # 2. Nạp trọng số checkpoint (nếu có và không phải là 'hf_base')
    if actual_checkpoint and actual_checkpoint.lower() != "hf_base":
        if os.path.exists(actual_checkpoint):
            load_checkpoint_weights(model, actual_checkpoint, device)
        else:
            print(f"Cảnh báo: Không tìm thấy checkpoint tại {actual_checkpoint}. Chạy trực tiếp model gốc HF!")

    model.to(device)
    model.eval()

    # 3. Nạp tập dữ liệu
    dataset_dict = load_from_disk(dataset_path)
    eval_data = dataset_dict[split_name]

    if max_samples is not None and max_samples < len(eval_data):
        eval_data = eval_data.select(range(max_samples))

    samples_count = len(eval_data)
    print(f"Tổng số mẫu đánh giá: {samples_count:,}")

    # 4. DataLoader với dynamic padding
    data_collator = DataCollatorForSeq2Seq(
        tokenizer=tokenizer,
        model=model,
        padding=True,
        label_pad_token_id=tokenizer.pad_token_id
    )

    eval_dataloader = DataLoader(
        eval_data,
        batch_size=batch_size,
        shuffle=False,
        collate_fn=data_collator
    )

    predictions = []
    references = []

    start_time = time.time()

    # 5. Vòng lặp suy luận
    with torch.inference_mode():
        for batch in tqdm(eval_dataloader, desc=f"Evaluating ({strategy_name})"):
            input_ids = batch["input_ids"].to(device, non_blocking=True)
            attention_mask = batch["attention_mask"].to(device, non_blocking=True)
            labels = batch["labels"].to(device, non_blocking=True)

            gen_kwargs = {
                "max_length": max_length,
                "num_beams": num_beams,
                "early_stopping": True if num_beams > 1 else False,
            }

            with torch.amp.autocast(device_type="cuda", dtype=torch.float16):
                generated_tokens = model.generate(
                    input_ids=input_ids,
                    attention_mask=attention_mask,
                    **gen_kwargs
                )

            decoded_preds = tokenizer.batch_decode(
                generated_tokens,
                skip_special_tokens=True
            )

            labels_np = labels.cpu().numpy()
            labels_np = np.where(labels_np != -100, labels_np, tokenizer.pad_token_id)
            decoded_labels = tokenizer.batch_decode(
                labels_np,
                skip_special_tokens=True
            )

            predictions.extend([pred.strip() for pred in decoded_preds])
            references.extend([[ref.strip()] for ref in decoded_labels])

    elapsed_time = time.time() - start_time
    speed = samples_count / elapsed_time if elapsed_time > 0 else 0.0

    # 6. Tính điểm SacreBLEU
    bleu_result = bleu_metric.compute(predictions=predictions, references=references)
    score_bleu = bleu_result["score"]

    print(f"Hoàn thành: {elapsed_time:.2f}s | Tốc độ: {speed:.1f} mẫu/s")
    print(f"SacreBLEU: {score_bleu:.2f}")

    del model, tokenizer
    torch.cuda.empty_cache()

    return {
        "model": model_name,
        "checkpoint": actual_checkpoint if actual_checkpoint else "HuggingFace_Base",
        "strategy": strategy_name,
        "split": split_name,
        "samples": samples_count,
        "bleu": round(score_bleu, 2),
        "time_sec": round(elapsed_time, 2),
        "speed_samples_per_sec": round(speed, 2)
    }

In [ ]:
PATH_EN2VI = "/kaggle/input/notebooks/khnhhngnguynhu/preprocess-phomt-mariannmt/PhoMT/datasets_en2vi"
PATH_VI2EN = "/kaggle/input/notebooks/khnhhngnguynhu/preprocess-phomt-mariannmt/PhoMT/datasets_vi2en"
CHECKPOINT_EN2VI = "/kaggle/input/notebooks/khnhhngnguynhu/finetune-en2vi-phonmt-mariannmt/checkpoints_marian_en2vi/checkpoint-15282"
experiments = [
    # 1. Chieu English -> Vietnamese
    {
        # "model_name": "Helsinki-NLP/opus-mt-en-vi",
        "model_name": CHECKPOINT_EN2VI,
        "dataset_path": PATH_EN2VI,
        "num_beams": 1  # Greedy Search
    },
    {
        # "model_name": "Helsinki-NLP/opus-mt-en-vi",
        "model_name": CHECKPOINT_EN2VI,
        "dataset_path": PATH_EN2VI,
        "num_beams": 4  # Beam Search
    },

    # # 2. Chieu Vietnamese -> English
    # {
    #     "model_name": "Helsinki-NLP/opus-mt-vi-en",
    #     "dataset_path": PATH_VI2EN,
    #     "num_beams": 1  # Greedy Search
    # },
    # {
    #     "model_name": "Helsinki-NLP/opus-mt-vi-en",
    #     "dataset_path": PATH_VI2EN,
    #     "num_beams": 4  # Beam Search
    # }
]

benchmarks = []

for experiment in experiments:
    results = evaluate_model(
        model_name=experiment["model_name"],
        dataset_path=experiment["dataset_path"],
        split_name="test",
        batch_size=64,
        max_length=128,
        num_beams=experiment["num_beams"],
        max_samples=None 
    )
    benchmarks.append(results)

# Hien thi bang tong hop ket qua
df_results = pd.DataFrame(benchmarks)
print("\n" + "_"*80)
print("BENCHMARK BASE MODELS")
print("_"*80)
display(df_results)


Model Base  : /kaggle/input/notebooks/khnhhngnguynhu/finetune-en2vi-phonmt-mariannmt/checkpoints_marian_en2vi/checkpoint-15282
Checkpoint  : /kaggle/input/models/khnhhngnguynhu/phomt-mariannmt/pytorch/vie2eng/2/vienv2.pt
Strategy    : Greedy Search
Data Path   : /kaggle/input/notebooks/khnhhngnguynhu/preprocess-phomt-mariannmt/PhoMT/datasets_en2vi [test]



Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

Đang nạp trọng số từ: /kaggle/input/models/khnhhngnguynhu/phomt-mariannmt/pytorch/vie2eng/2/vienv2.pt
Nạp checkpoint thành công! (Missing keys: 259, Unexpected keys: 128)
Tổng số mẫu đánh giá: 19,151


Evaluating (Greedy Search):   0%|          | 0/300 [00:00<?, ?it/s]/tmp/ipykernel_49/2788242097.py:144: UserWarning: CUDA is not available or torch_xla is imported. Disabling autocast.
  with torch.amp.autocast(device_type="cuda", dtype=torch.float16):
Evaluating (Beam Search (beams=4)):  63%|██████▎   | 190/300 [1:24:15<2:42:29, 88.63s/it]

In [ ]:
# sample_en = [
#     "It begins with a countdown.",
#     "Across India, people hold their breath for the declaration of independence.",
#     "Artificial intelligence is rapidly transforming global technology."
# ]

# tok_en2vi = AutoTokenizer.from_pretrained("Helsinki-NLP/opus-mt-en-vi")
# mdl_en2vi = AutoModelForSeq2SeqLM.from_pretrained("Helsinki-NLP/opus-mt-en-vi").to(device)

# inputs = tok_en2vi(sample_en, return_tensors="pt", padding=True, truncation=True).to(device)

# # Sinh bang Greedy
# with torch.inference_mode():
#     out_greedy = mdl_en2vi.generate(**inputs, max_length=128, num_beams=1)
#     out_beam = mdl_en2vi.generate(**inputs, max_length=128, num_beams=4, early_stopping=True)

# trans_greedy = tok_en2vi.batch_decode(out_greedy, skip_special_tokens=True)
# trans_beam = tok_en2vi.batch_decode(out_beam, skip_special_tokens=True)

# for i in range(len(sample_en)):
#     print(f"\n[EN]: {sample_en[i]}")
#     print(f" -> Greedy Search: {trans_greedy[i]}")
#     print(f" -> Beam Search  : {trans_beam[i]}")